# 06. Dynamic Resource Scheduler & Sampling Control
Demonstrates `ResourceScheduler`'s direction-weighted batch sampling (per `configs/training/multilingual.yaml`) and `MultilingualPairMixer`'s weighted-mixture dataset construction.

In [1]:
# ============================================================
# PATH BOOSTER -- guarantees project root in sys.path & CWD.
# Matches notebooks/00_setup_environment.ipynb's convention: this repo
# is deployed both to Colab (fresh `git clone`, folder "Ekegusii-LLM-Translation")
# and to Kineses Cloud / similar Jupyter hosts (pre-placed at
# ~/Ekegusii-LLM-Translation-main -- the "-main" suffix comes from GitHub's
# "Download ZIP" naming). Do not assume either folder name is the cwd.
# ============================================================
import os
import sys

REPO_NAME = "Ekegusii-LLM-Translation"


def _find_project_root():
    if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
        if not os.path.exists(REPO_NAME):
            os.system(f"git clone https://github.com/aykahsay/{REPO_NAME}.git")
            os.system(f"pip install -q -r {REPO_NAME}/requirements.txt")
        return os.path.abspath(REPO_NAME)

    try:
        cwd = os.getcwd()
    except FileNotFoundError:
        cwd = os.path.expanduser("~")
        os.chdir(cwd)

    home = os.path.expanduser("~")
    for candidate in (f"{REPO_NAME}-main", REPO_NAME):
        proj_dir = os.path.join(home, candidate)
        if os.path.isdir(proj_dir):
            return proj_dir

    if os.path.exists("src") and os.path.exists("data"):
        return cwd
    if os.path.basename(cwd) == "notebooks" and os.path.exists(os.path.join("..", "src")):
        return os.path.abspath("..")

    return cwd


project_root = _find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Project root: {project_root}")


Project root: C:\Users\Admin\OneDrive - United States International University (USIU)\Documents\NLP\Multilogual_transaltion_nlp


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
from src.utils.config_dict import load_yaml
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.scheduler import ResourceScheduler
from src.task_generation.translation_pairs import direction_counts
from src.task_generation.multilingual_pairs import MultilingualPairMixer
from src.utils.constants import LANGUAGE_CODES

manager = MasterCorpusManager()
train_df = manager.load_train_split()
multilingual_cfg = load_yaml('configs/training/multilingual.yaml')
multilingual_cfg

INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


{'languages': ['eng', 'swh', 'eke'],
 'direction_weights': {'eng_to_eke': 1.0,
  'eke_to_eng': 1.0,
  'swa_to_eke': 1.0,
  'eke_to_swa': 1.0,
  'eng_to_swa': 0.5,
  'swa_to_eng': 0.5},
 'sampling_temperature': 1.5,
 'lexical_augmentation': {'enabled': False, 'mix_ratio': 0.1}}

## Per-direction sampling probabilities

In [4]:
counts = direction_counts(train_df)
counts_by_key = {
    f'{LANGUAGE_CODES[src]}_to_{LANGUAGE_CODES[tgt]}': count
    for label, count in counts
    for src, tgt in [label.split('->')]
}
scheduler = ResourceScheduler(multilingual_cfg)
probs = scheduler.compute_direction_sampling_probs(counts_by_key)
probs

INFO | Extracted 37,721 'English'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 37,721 'Ekegusii'->'English' pairs from 39,421 rows.


INFO | Extracted 27,092 'Kiswahili'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 27,092 'Ekegusii'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'English'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'Kiswahili'->'English' pairs from 39,421 rows.


INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.22526608467210832, 'eke_to_eng': 0.22526608467210832, 'swa_to_eke': 0.180662378496575, 'eke_to_swa': 0.180662378496575, 'eng_to_swa': 0.09407153683131678, 'swa_to_eng': 0.09407153683131678}


{'eng_to_eke': 0.22526608467210832,
 'eke_to_eng': 0.22526608467210832,
 'swa_to_eke': 0.180662378496575,
 'eke_to_swa': 0.180662378496575,
 'eng_to_swa': 0.09407153683131678,
 'swa_to_eng': 0.09407153683131678}

## Simulated batch quota (batch_size=64)

In [5]:
scheduler.build_mixed_batch_plan(counts_by_key, batch_size=64)

INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.22526608467210832, 'eke_to_eng': 0.22526608467210832, 'swa_to_eke': 0.180662378496575, 'eke_to_swa': 0.180662378496575, 'eng_to_swa': 0.09407153683131678, 'swa_to_eng': 0.09407153683131678}


{'eng_to_eke': 17,
 'eke_to_eng': 13,
 'swa_to_eke': 14,
 'eke_to_swa': 11,
 'eng_to_swa': 6,
 'swa_to_eng': 3}

## Weighted-mixture dataset (sample)

In [6]:
sample_df = train_df.sample(500, random_state=42)
mixer = MultilingualPairMixer(multilingual_cfg)
mixture = mixer.build_weighted_mixture(sample_df, target_total=200)
mixture['source_lang'].str.cat(mixture['target_lang'], sep='->').value_counts()

INFO | Extracted 476 'English'->'Ekegusii' pairs from 500 rows.


INFO | Extracted 476 'Ekegusii'->'English' pairs from 500 rows.


INFO | Extracted 345 'Kiswahili'->'Ekegusii' pairs from 500 rows.


INFO | Extracted 345 'Ekegusii'->'Kiswahili' pairs from 500 rows.


INFO | Extracted 369 'English'->'Kiswahili' pairs from 500 rows.


INFO | Extracted 369 'Kiswahili'->'English' pairs from 500 rows.


INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.2243342186761914, 'eke_to_eng': 0.2243342186761914, 'swa_to_eke': 0.1810104376273729, 'eke_to_swa': 0.1810104376273729, 'eng_to_swa': 0.09465534369643568, 'swa_to_eng': 0.09465534369643568}


INFO | Built weighted mixture: 200 rows (target was 200).


source_lang
English->Ekegusii      51
Ekegusii->Kiswahili    40
Ekegusii->English      33
Kiswahili->Ekegusii    32
Kiswahili->English     23
English->Kiswahili     21
Name: count, dtype: int64